# Answer Feedback Tutor — Colab version

Runs the same idea as the local app (local LLM grading + local image generation, no cloud APIs),
adapted to run inside Google Colab:

- **Ollama** runs as a background process for the LLM grading step (works on free Colab).
- **Stable Diffusion** is loaded directly via the `diffusers` library instead of AUTOMATIC1111's
  web UI, because Google blocks the Gradio-based web UI on the free tier — the model itself is
  not blocked, only that particular server/UI.

**Runtime > Change runtime type > GPU (T4 is fine)** before running any cell.

> Note: this notebook is for development/demo convenience because Colab gives you free GPU access.
> Your submission should still note that the app also runs on a local machine — see the main
> README for the local (Ollama + AUTOMATIC1111) version. The core grading and prompt-building
> logic here is the same either way.


In [ ]:
# 1. Install and start Ollama in the background
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)  # give the server a moment to start

!ollama pull llama3
print("Ollama is running.")


In [ ]:
# 2. Install diffusers and dependencies for local image generation
!pip install -q diffusers transformers accelerate safetensors


In [ ]:
# 3. Load a local Stable Diffusion pipeline (downloads weights once, then cached)
import torch
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"  # swap for any local/open checkpoint you prefer
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")
print("Stable Diffusion pipeline loaded.")


In [ ]:
# 4. Rubric data (same content as data/rubrics/sample_questions.json)
rubrics = [
    {
        "id": "photosynthesis",
        "question": "Explain the process of photosynthesis in plants.",
        "key_points": [
            {"concept": "Occurs in chloroplasts using chlorophyll"},
            {"concept": "Light-dependent reactions split water and release oxygen"},
            {"concept": "Carbon dioxide is used to produce glucose"},
            {"concept": "Light-independent reactions (Calvin cycle)"},
            {"concept": "Glucose is used for plant energy and growth"},
        ],
    },
    {
        "id": "neuron_signal",
        "question": "Describe how a neuron transmits a nerve signal.",
        "key_points": [
            {"concept": "Resting potential from ion concentration differences"},
            {"concept": "Depolarization via sodium ion influx"},
            {"concept": "Action potential travels along the axon"},
            {"concept": "Neurotransmitters released at the synapse"},
            {"concept": "Neurotransmitters bind to receptors on next neuron"},
        ],
    },
    {
        "id": "newtons_second_law",
        "question": "Explain Newton's second law of motion.",
        "key_points": [
            {"concept": "F = ma relationship"},
            {"concept": "Acceleration proportional to net force"},
            {"concept": "Acceleration inversely proportional to mass"},
            {"concept": "Force and acceleration are vector quantities"},
        ],
    },
]
questions_by_id = {r["id"]: r for r in rubrics}


In [ ]:
# 5. Grading function — same prompt strategy as src/grading.py, calling local Ollama
import json, re, requests

GRADING_PROMPT_TEMPLATE = '''You are an exam grader. Compare the STUDENT ANSWER to the QUESTION and the
RUBRIC below. The rubric is a list of key concepts that a complete answer should cover.

QUESTION:
{question}

RUBRIC (key concepts expected in a complete answer):
{rubric_list}

STUDENT ANSWER:
{student_answer}

Decide, for each rubric concept, whether the student's answer covers it (even if worded
differently), or misses it. Then respond with ONLY a JSON object in exactly this shape,
with no extra commentary, no markdown fences, and no text before or after it:

{{
  "score": <number 0-100>,
  "covered_points": ["<concept text>", ...],
  "missed_points": ["<concept text>", ...],
  "summary": "<one or two sentence overall feedback>"
}}
'''

def grade_answer(question, key_points, student_answer, model="llama3", retries=1):
    rubric_list = "\n".join(f"- {kp['concept']}" for kp in key_points)
    prompt = GRADING_PROMPT_TEMPLATE.format(question=question, rubric_list=rubric_list, student_answer=student_answer)

    for _ in range(retries + 1):
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=120,
        )
        response.raise_for_status()
        raw_text = response.json().get("response", "")
        match = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if match:
            try:
                result = json.loads(match.group(0))
                result.setdefault("covered_points", [])
                result.setdefault("missed_points", [])
                result.setdefault("summary", "")
                result.setdefault("score", 0)
                return result
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Model did not return valid JSON. Last raw output:\n{raw_text}")


In [ ]:
# 6. Image generation function — builds a prompt from missed points, runs it through diffusers
def build_image_prompt(missed_points, topic):
    if not missed_points:
        return ""
    concepts = ", ".join(missed_points[:3])
    return (
        f"simple educational diagram illustrating {topic}, focusing on {concepts}, "
        f"clean labeled infographic style, white background, textbook illustration"
    )

def generate_image(prompt, steps=25):
    if not prompt:
        return None
    image = pipe(prompt, num_inference_steps=steps, guidance_scale=7).images[0]
    return image


In [ ]:
# 7. Simple in-notebook UI with ipywidgets
import ipywidgets as widgets
from IPython.display import display, clear_output

question_dropdown = widgets.Dropdown(
    options=[(r["question"], r["id"]) for r in rubrics],
    description="Question:",
    layout=widgets.Layout(width="600px"),
)
answer_box = widgets.Textarea(
    placeholder="Type your answer here...",
    layout=widgets.Layout(width="600px", height="150px"),
)
submit_button = widgets.Button(description="Grade my answer", button_style="primary")
output_area = widgets.Output()

def on_submit(_):
    with output_area:
        clear_output()
        answer = answer_box.value.strip()
        if not answer:
            print("Please type an answer first.")
            return

        selected = questions_by_id[question_dropdown.value]
        print("Grading with local LLM...")
        result = grade_answer(selected["question"], selected["key_points"], answer)

        print(f"\nScore: {result['score']}/100")
        print(result["summary"])
        print("\nCovered points:")
        for p in result["covered_points"]:
            print(f"  - {p}")
        print("\nMissed points:")
        for p in result["missed_points"]:
            print(f"  - {p}")

        if result["missed_points"]:
            print("\nGenerating a study diagram for what you missed...")
            prompt = build_image_prompt(result["missed_points"], topic=selected["question"])
            image = generate_image(prompt)
            display(image)
        else:
            print("\nYou covered every key point — no gaps to illustrate!")

submit_button.on_click(on_submit)
display(question_dropdown, answer_box, submit_button, output_area)
